# Colab fallback — deepfake-detection

Use this only if local training on the 6GB RTX 3060 is too slow or OOMs.
Same code, same commands as local — just a different machine.

**Before using this notebook:**
1. Preprocess face crops locally first (fast enough on your machine) — do NOT re-run MTCNN extraction on Colab, it wastes session time.
2. Upload the resulting `ffpp_faces/` folder and `splits/` folder to Google Drive.
3. Either push this repo to a private GitHub repo (preferred, so `--resume` checkpoints and code stay in sync), or zip the repo folder and upload it to Drive alongside the data.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Option A — clone from your own GitHub repo
Replace `<your-repo-url>` below once you've pushed this project.

In [ ]:
!git clone <your-repo-url> deepfake-detection
%cd deepfake-detection
!pip install -q -r requirements.txt

## Option B — unzip a copy you uploaded to Drive instead
Skip Option A above if you use this.

In [ ]:
!unzip -q /content/drive/MyDrive/deepfake-detection.zip -d /content/
%cd /content/deepfake-detection
!pip install -q -r requirements.txt

## Train (resumable — safe to interrupt / session timeout)
Keep `--out` on Drive so checkpoints survive disconnects. Edit `--model` / `--methods` / `--compression` per run, matching `scripts/run_all_experiments.sh`.

In [ ]:
!python -m src.train \
  --data-root /content/drive/MyDrive/ffpp_faces \
  --splits-dir /content/drive/MyDrive/splits \
  --model xception --methods Deepfakes --compression c23 \
  --out /content/drive/MyDrive/runs/xception_DF_c23 --resume

## Evaluate

In [ ]:
!python -m src.evaluate \
  --checkpoint /content/drive/MyDrive/runs/xception_DF_c23/best.pt \
  --data-root /content/drive/MyDrive/ffpp_faces \
  --splits-dir /content/drive/MyDrive/splits \
  --methods Face2Face --compression c23 \
  --out /content/drive/MyDrive/results/xception_DF_to_F2F_c23.json

## Copy results back for local analysis
Run `python -m analyze.analyze` locally, pointed at a `results/` folder synced down from Drive — no need to run analysis on Colab.